# NISAR L-band SAR + Sentinel-1 + DEM (+ optional Sentinel-2) — Arctic calving-glacier datacube

<a href="https://colab.research.google.com/github/buckai-observatory/geoai-datacubes/blob/feature/earth-engine-provider/notebooks/05_nisar_arctic_datacube.ipynb" target="_blank" rel="noopener noreferrer"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>
<!-- BRANCH-PREVIEW: swap `feature/earth-engine-provider` -> `main` at merge time -->


Composes a data cube of **NISAR L-band SAR**
(`NISAR_L2_GCOV_PROVISIONAL_V1`, Geocoded Polarimetric Covariance),
**Sentinel-1 C-band SAR**, **ArcticDEM** (default) or **Copernicus DEM**,
and **Sentinel-2** optical (when a cloud-free scene exists) over an
**Arctic calving-glacier / ice-cap AOI**. Uses the pipeline's new
`earthdata` provider (talking to NASA CMR / Earthdata Cloud via
`earthaccess`) alongside the existing STAC providers and `direct_http`
to demonstrate that `geoai-datacubes` composes cutting-edge public EO
data (**NISAR L-band went public 2026-07-20**, ~7 weeks before this
notebook was written) with established archives (S1 back to 2014, S2
back to 2015) into a single ML-ready UTM cube.

**AOI picker:** NISAR's polar orbit visits any given AOI only every
few days. The setup cell searches NISAR CMR for granules near a
**hint** `(TARGET_LAT, TARGET_LON)`, scores each granule's coverage
and polarisation availability, and either keeps the hint AOI or
re-centres it on the best-covered granule's centroid. Either way
you get a substantially covered NISAR scene without hand-tuning
coordinates. The **AOI is specified in exactly one place** — the
setup cell below — so the rest of the notebook stays generic across
Arctic targets.

**Polarisation availability caveat:** NISAR observation modes vary
by region and by orbit pass. Asking for `bands=["HH", "HV"]` returns
whatever the specific granule the picker landed on actually contains
— some regions currently have **single-pol scenes only** (typically
`SHNA` = HH-only, `NASV` = VV-only), so HV can come back as NaN
without warning if the picker had no dual-pol candidates. The
default target below is picked from a region with confirmed dual-pol
(DHDH = HH+HV) coverage in the current archive, but this changes as
NISAR fills its 12-day repeat cycle. Watch the NISAR fetch cell's
output for the `[HV] not present in this granule` line to confirm.

Prerequisites:

- A NASA Earthdata Login (free, one-time signup at
  <https://urs.earthdata.nasa.gov/users/new>). Needed for the NISAR
  fetch through `earthaccess`.
- The `earthengine` install extra (matches notebook 04) plus
  `earthaccess`, `h5py`, and `shapely` — the Colab cell below handles
  this: `pip install geoai-datacubes[earthengine,ml] earthaccess h5py shapely`.
- Two `EARTHDATA_USERNAME` + `EARTHDATA_PASSWORD` Colab secrets —
  **remember to toggle "Notebook access" ON for these two secrets
  specifically on this notebook** (they don't inherit from notebook
  04's grants). Legacy `EDL_USERNAME` / `EDL_PASSWORD` names are also
  accepted and auto-remapped. The bootstrap prints exactly which
  secrets it found; if you see `Colab secret MISSING: EARTHDATA_USERNAME`,
  that's the fix.
- Optionally, the same `EARTHENGINE_TOKEN` + `EARTHENGINE_PROJECT`
  Colab secrets as notebook 04 (kept for parity even though this
  notebook does not currently touch Earth Engine directly).

See [`docs/providers/earthdata.md`](../docs/providers/earthdata.md)
for the full auth / product-inventory / caveats reference.


In [ ]:
# --- Colab / local bootstrap ---
# On Colab, this cell clones the repo, installs geoai-datacubes with the
# [earthengine,ml] extras plus `earthaccess` + `h5py` (for the NISAR L2
# GCOV HDF5 files), and reads up to four optional Colab userdata
# secrets:
#
#   EARTHENGINE_TOKEN   -- persisted-credentials JSON from a machine that has
#                          already run `ee.Authenticate()`. Kept for parity
#                          with notebook 04 even though this notebook does
#                          not touch Earth Engine directly.
#   EARTHENGINE_PROJECT -- your GCP project ID with the EE API enabled.
#   EDL_USERNAME        -- your NASA Earthdata Login username.
#   EDL_PASSWORD        -- your NASA Earthdata Login password.
#                          The two EDL_* secrets let `earthaccess.login(
#                          strategy="environment")` succeed without any
#                          browser popup / prompt. If they are absent we
#                          fall back to `strategy="interactive"`.
#
# COMMON GOTCHA: Colab requires "Notebook access" toggled ON per secret
# PER NOTEBOOK. If you already set the secrets for notebook 04 but see
# an interactive Earthdata prompt below, you need to toggle Notebook
# access for EDL_USERNAME + EDL_PASSWORD *on this specific notebook*.
# The bootstrap prints exactly which secrets it found so it's obvious
# when one is missing / not accessible.
#
# BRANCH-PREVIEW: while feature/earth-engine-provider is still unmerged
# we clone that branch so the [earthdata] provider + NISAR-L mission
# profile exist in the checkout. Swap `BRANCH` back to "main" at merge
# time.
BRANCH   = "feature/earth-engine-provider"
REPO_URL = "https://github.com/buckai-observatory/geoai-datacubes.git"

import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _run(cmd, label):
    """Run a subprocess with stderr surfaced -- Colab sometimes swallows
    subprocess stderr, and a bare `check_call` failure gives you exit 128
    with no clue what went wrong."""
    print(f"  $ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.rstrip())
    if r.stderr.strip():
        # git normally writes progress to stderr, so this is not always an error.
        print(r.stderr.rstrip())
    if r.returncode != 0:
        raise RuntimeError(f"{label} failed with exit {r.returncode}")


def _clone_branch(repo_url, branch, dest):
    """Fresh clone of ``branch`` into ``dest``. Prefers a shallow single-branch
    clone (fast, small); falls back to a full clone + branch checkout if the
    shallow-branch path errors out (some Colab git versions have been observed
    to 500 on --branch <b> with slashes in the branch name)."""
    try:
        _run(
            ["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(dest)],
            f"shallow clone of branch {branch!r}",
        )
        return
    except RuntimeError as exc:
        print(f"  (shallow branch clone failed: {exc})")
        print("  falling back to full clone + checkout")
    if dest.exists():
        shutil.rmtree(dest)
    _run(["git", "clone", repo_url, str(dest)], "full clone")
    _run(["git", "-C", str(dest), "checkout", branch], f"checkout of {branch!r}")


def _pick_up_colab_secrets(names):
    """Copy any available Colab userdata secrets into os.environ, and PRINT
    diagnostics for both found AND missing ones so it's obvious when a
    'Notebook access' toggle needs flipping."""
    try:
        from google.colab import userdata
    except Exception:
        return
    for secret in names:
        try:
            val = userdata.get(secret)
        except Exception:
            val = None
        if val:
            os.environ[secret] = val
            print(f"  Colab secret found:   {secret}")
        else:
            print(f"  Colab secret MISSING: {secret}  "
                  f"(set it via the key icon in the left sidebar, and toggle "
                  f"'Notebook access' ON *for this notebook*)")


if IN_COLAB:
    print(f"Colab detected -- bootstrapping repo + dependencies (branch: {BRANCH})")
    REPO_DIR = Path("/content/geoai-datacubes")
    if REPO_DIR.exists():
        print(f"  removing stale checkout at {REPO_DIR} to force a fresh clone")
        shutil.rmtree(REPO_DIR)
    _clone_branch(REPO_URL, BRANCH, REPO_DIR)

    # Install: fresh geoai-datacubes source + extras + earthaccess/h5py for
    # NISAR + contextily (used elsewhere in this pipeline; small, no-op if
    # already present) + shapely (for the NISAR granule-footprint AOI
    # helper below; usually already there but explicit is safe).
    _run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--force-reinstall", "--no-deps", str(REPO_DIR)],
        "pip install (fresh geoai-datacubes source)",
    )
    _run(
        [sys.executable, "-m", "pip", "install", "-q",
         f"{REPO_DIR}[earthengine,ml]", "earthaccess", "h5py", "shapely"],
        "pip install [earthengine,ml] extras + earthaccess + h5py + shapely",
    )

    os.chdir(REPO_DIR / "notebooks")
    print(f"cwd = {os.getcwd()}")

    print("\n--- Checking Colab userdata secrets ---")
    _pick_up_colab_secrets(
        ["EARTHENGINE_TOKEN", "EARTHENGINE_PROJECT",
         # earthaccess's canonical env var names, and legacy fallbacks:
         "EARTHDATA_USERNAME", "EARTHDATA_PASSWORD",
         "EDL_USERNAME", "EDL_PASSWORD"]
    )
    # Re-map legacy EDL_* names to earthaccess's canonical EARTHDATA_* so
    # `strategy="environment"` picks them up. (Earlier docs of ours told
    # users to name the secrets EDL_USERNAME / EDL_PASSWORD -- those get
    # accepted here for backward compat but forwarded to the names
    # earthaccess actually reads.)
    if not os.environ.get("EARTHDATA_USERNAME") and os.environ.get("EDL_USERNAME"):
        os.environ["EARTHDATA_USERNAME"] = os.environ["EDL_USERNAME"]
    if not os.environ.get("EARTHDATA_PASSWORD") and os.environ.get("EDL_PASSWORD"):
        os.environ["EARTHDATA_PASSWORD"] = os.environ["EDL_PASSWORD"]
else:
    print("Local environment -- using existing checkout")
    print("Assuming Earthdata is authenticated (see docs/providers/earthdata.md)")


# ---- Earthdata authentication -----------------------------------------
# earthaccess picks up EDL_USERNAME + EDL_PASSWORD via
# strategy="environment"; if those are missing we fall back to
# strategy="interactive" which prompts for username + password inline.
# The interactive path works fine but is one-time-per-runtime; the
# secret path is one-time-forever, so it's worth setting the secrets
# if you're going to re-run this notebook.
try:
    import earthaccess
    if os.environ.get("EARTHDATA_USERNAME") and os.environ.get("EARTHDATA_PASSWORD"):
        earthaccess.login(strategy="environment")
        print("\nEarthdata login: environment strategy OK (using EARTHDATA_USERNAME/EARTHDATA_PASSWORD)")
    else:
        print("\n(no EARTHDATA_USERNAME/EARTHDATA_PASSWORD env vars found -- prompting interactively)")
        print("If you set them as Colab secrets, make sure 'Notebook access' is toggled ON for this notebook.")
        print("(Legacy EDL_USERNAME/EDL_PASSWORD names are also accepted and auto-remapped above.)")
        earthaccess.login(strategy="interactive")
        print("Earthdata login: interactive strategy OK")
except Exception as exc:
    print(f"(Earthdata login skipped: {type(exc).__name__}: {exc})")
    print("The NISAR fetch cell will fail with an auth error until this is fixed.")


## Setup

Configure the target Arctic AOI (set in exactly one place -- the cell below), the fetch window, and the master grid resolution used for every scene. The AOI is expressed as a `(TARGET_LAT, TARGET_LON)` + `RADIUS_KM` pair so it is trivial to move the demo to any Arctic target -- just override the four constants at the top of the code cell.

The window starts at `2026-06-17`, the NISAR public-data release date, and runs to the end of the first publicly-available cycle. Sentinel-1, ArcticDEM, and Copernicus DEM all cover the same window (S1 collects most Arctic AOIs every 6-12 days, DEMs are static). Sentinel-2 is optional and often comes up empty over polar coastal glaciers due to persistent cloud -- the fetch cell handles that gracefully.


In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import rasterio
import matplotlib as mpl
import matplotlib.pyplot as plt

# locate the repo root from this notebook's CWD (notebooks/)
NB_DIR = Path.cwd()
if NB_DIR.name != "notebooks":
    for p in (NB_DIR, *NB_DIR.parents):
        if (p / "notebooks").is_dir() and (p / "geoai_datacubes").is_dir():
            NB_DIR = p / "notebooks"
            break
REPO_ROOT = NB_DIR.parent
sys.path.insert(0, str(REPO_ROOT))

# Scratch folders for this notebook.
OUT  = NB_DIR / "_outputs_nb05"
DATA = OUT / "data"
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# Target Arctic AOI (the one place a specific location is set)
# ============================================================
# Default target: northern Baffin Island plateau (Nunavut, Canadian
# Arctic), on the Steacie-Sirmilik-Piling icefield uplands. Coordinates
# chosen empirically from NISAR CMR + Planetary Computer S1 STAC
# queries to guarantee:
#   * >= 2 NISAR L2 GCOV granules FULLY covering the 5-km AOI in
#     DUAL-POL mode (DHDH = HH + HV), so both L-band polarisations
#     come back real -- and the picker's polarisation preference will
#     select DHDH rather than falling back to single-pol NASV.
#   * >= 76 Sentinel-1 RTC HH+HV dual-pol scenes in the 2024-2026
#     window, so the L-vs-C-band comparison is guaranteed to work
#     regardless of Colab-side transient network issues.
#   * Well inside ArcticDEM v4.1's coverage (all latitudes >60N).
#   * On solid Baffin Island land, not sea -- the OSM panel below
#     the setup will confirm.
#
# To swap in a different Arctic target, change TARGET_LAT / TARGET_LON
# below -- everything else in the notebook stays generic. Alternatives
# (each has caveats -- see comment for details):
#   Cumberland Peninsula (66.6, -65.5)  -- 1 DHDH granule available;
#                                          picker may drift if it clips.
#   Bylot / Sirmilik NP  (72.9, -78.5)  -- 3 DHDH NISAR but S1 only HH.
#   Penny Ice Cap        (67.25, -65.75) -- Only 1 DHDH; picker often drifts.
#   Helheim Glacier      (66.40, -38.10) -- SE Greenland; NISAR SP-HH only.
#   Jakobshavn Isbrae    (69.15, -50.00) -- W Greenland fastest glacier; SP-HH only.
#
# Search radius (200-300 km) controls how far the granule-footprint
# picker looks. RADIUS_KM sets AOI size; NISAR frames typically have
# ~7-10 km along-track valid extents after the polygon-declared swath
# edges taper off, so a 5 km radius (10 x 10 km AOI) fits cleanly
# inside a single frame with room to spare.
TARGET_LAT, TARGET_LON = 71.6, -72.75
TARGET_NAME            = "northern Baffin Island plateau, Canadian Arctic"
RADIUS_KM              = 5.0
SEARCH_RADIUS_KM       = 250.0                # how far around the hint to search
TIME_RANGE             = ("2026-06-17", "2026-08-05")   # since NISAR public release
RESOLUTION             = 10                              # metres -- Sentinel-2 RGB native; upsample S1/NISAR/ArcticDEM to match
ARCTICDEM_RESOLUTION   = "10m"                           # PGC v4.1 mosaic tile resolution: "2m" | "10m" | "32m" | "100m" | "500m". "10m" matches the cube; "32m" is faster/smaller but coarser.
MAX_CLOUD              = 0.30                            # S2 will be rare at high latitudes; be tolerant
INCLUDE_S2             = True                            # optional -- toggle off if you don't want optical
DEM_MISSION            = "ArcticDEM"                     # "ArcticDEM" (>60N; PGC) or "Copernicus-DEM" (global fallback)
# Widen the S1 and S2 look-back windows separately: their coverage
# over Arctic AOIs is sparse relative to NISAR's 12-day cycle
# (Sentinel-1 RTC has patchy dual-pol availability; Sentinel-2
# demands cloud-free scenes which are rare in polar summers). Ice
# caps change slowly enough that scenes from earlier in the year --
# or even a year or two back -- are still valid C-band / optical
# comparators to NISAR L-band. Tighten these back down if you're
# doing per-date change detection instead.
S1_TIME_RANGE          = ("2024-01-01", "2026-08-05")
S2_TIME_RANGE          = ("2024-06-01", "2026-08-05")


def _bbox_from_center_radius(lat, lon, r_km):
    """Return [lon_min, lat_min, lon_max, lat_max] for a bbox with edge
    half-length r_km, centred at (lat, lon). Local flat-earth approximation."""
    dlat = r_km / 111.0
    dlon = r_km / (111.0 * np.cos(np.deg2rad(lat)))
    return [lon - dlon, lat - dlat, lon + dlon, lat + dlat]


def _pick_nisar_covered_aoi(target_lat, target_lon, radius_km,
                             time_range, search_radius_km=200.0):
    """Return an AOI whose declared footprint is well covered by at least
    one NISAR granule in the requested time window.

    The declared-footprint check is a *necessary* condition for good
    coverage but not sufficient: NISAR frames can taper off inside the
    declared polygon (see the module comment above). So this helper
    gets you into a granule's swath; the provider then sorts by
    footprint overlap again and picks the best one. If the final
    reprojected raster still shows a lot of NaN, shrink `RADIUS_KM`.

    Requires shapely.
    """
    import earthaccess
    from shapely.geometry import Polygon, box

    search_bbox = _bbox_from_center_radius(target_lat, target_lon, search_radius_km)
    results = earthaccess.search_data(
        short_name="NISAR_L2_GCOV_PROVISIONAL_V1",
        bounding_box=tuple(search_bbox),
        temporal=time_range,
        count=25,
    )
    if not results:
        raise RuntimeError(
            f"No NISAR granules within {search_radius_km:.0f} km of "
            f"({target_lat:.2f}, {target_lon:.2f}) in {time_range}. "
            "Try widening SEARCH_RADIUS_KM or TIME_RANGE."
        )

    hint_box = box(*_bbox_from_center_radius(target_lat, target_lon, radius_km))
    scored = []   # (granule, poly, frac_at_hint, frac_at_centroid, centroid_lat, centroid_lon)
    for g in results:
        try:
            pts = (g["umm"]["SpatialExtent"]["HorizontalSpatialDomain"]
                    ["Geometry"]["GPolygons"][0]["Boundary"]["Points"])
            poly = Polygon([(p["Longitude"], p["Latitude"]) for p in pts])
            if not poly.is_valid or poly.is_empty:
                continue
            frac_hint = hint_box.intersection(poly).area / hint_box.area
            c = poly.centroid
            cbox = box(*_bbox_from_center_radius(c.y, c.x, radius_km))
            frac_c = cbox.intersection(poly).area / cbox.area
            pol_code = ""
            try:
                import re as _re
                m = _re.search(r"_(\w{4})_[AM]_", g["meta"]["native-id"])
                if m:
                    pol_code = m.group(1)
            except Exception:
                pass
            # Polarization rank: quad > dual > single > unknown. Rough
            # heuristic on the 4-char pol code in the granule ID
            # (SHNA=SP-HH, SVNA=SP-VV, DHDH=DP-HH+HV, DVDV=DP-VV+VH, ...).
            if pol_code.startswith("Q") or pol_code.endswith("Q"):
                pol_rank = 3
            elif pol_code.startswith("D"):
                pol_rank = 2
            elif pol_code.startswith("S"):
                pol_rank = 1
            else:
                pol_rank = 0
            scored.append((g, poly, frac_hint, frac_c, c.y, c.x, pol_rank))
        except (KeyError, IndexError, TypeError):
            continue

    if not scored:
        raise RuntimeError("Could not extract usable footprints from any granule.")

    # Sort by (coverage descending, pol_rank descending) so we prefer
    # dual/quad-pol granules when coverage ties. If HV was requested
    # (default) this dramatically increases the chance of getting both
    # bands non-NaN.
    scored.sort(key=lambda x: (-x[2], -x[6]))
    best_hint = scored[0]
    if best_hint[2] >= 0.90:
        g, _, frac, _, _, _, pol_rank = best_hint
        gid = g["meta"]["native-id"]
        aoi = _bbox_from_center_radius(target_lat, target_lon, radius_km)
        print(f"NISAR AOI picker: hint has {100*frac:.0f}% *declared* coverage in granule")
        print(f"                  {gid}")
        print(f"                  keeping AOI centred on your target.")
        return aoi

    scored.sort(key=lambda x: (-x[3], -x[6]))
    g, _, frac_hint, frac_c, clat, clon, _ = scored[0]
    gid = g["meta"]["native-id"]
    aoi = _bbox_from_center_radius(clat, clon, radius_km)
    print(f"NISAR AOI picker: hint only has {100*frac_hint:.0f}% NISAR coverage;")
    print(f"                  re-centring the AOI on granule centroid "
          f"({clat:.3f}, {clon:.3f}) ({100*frac_c:.0f}% declared coverage).")
    print(f"                  granule: {gid}")
    return aoi


# Resolve the AOI. Live CMR query -- earthaccess.login() must already
# have succeeded in the bootstrap cell above.
AOI = _pick_nisar_covered_aoi(
    TARGET_LAT, TARGET_LON, RADIUS_KM, TIME_RANGE,
    search_radius_km=SEARCH_RADIUS_KM,
)

aoi_lat = 0.5 * (AOI[1] + AOI[3])
aoi_lon = 0.5 * (AOI[0] + AOI[2])

print(f"\nAOI          : [{AOI[0]:.4f}, {AOI[1]:.4f}, {AOI[2]:.4f}, {AOI[3]:.4f}]")
print(f"AOI centre   : ({aoi_lat:.4f}, {aoi_lon:.4f})  radius {RADIUS_KM:.1f} km")
print(f"Target hint  : ({TARGET_LAT:.4f}, {TARGET_LON:.4f})  {TARGET_NAME}")
print(f"Time range   : {TIME_RANGE}")
print(f"Resolution   : {RESOLUTION} m")
print(f"Include S2   : {INCLUDE_S2}  (max cloud {MAX_CLOUD:.0%})")
print(f"Output dir   : {DATA}")


# ============================================================
# Journal-figure rcParams: bold labels, thick axes, no title.
# Note: neither `savefig.bbox="tight"` nor `savefig.dpi=300` are set
# here even though they are on the user's global defaults -- both
# interact badly with Colab's inline PNG renderer (bbox="tight" fights
# constrained_layout, and dpi=300 makes the browser downscale every
# image on layout / scroll passes, producing visible size oscillation).
# For paper export, pass `bbox_inches="tight", dpi=300` to individual
# savefig() calls instead of setting them globally in an interactive
# notebook.
# ============================================================
mpl.rcParams.update({
    "font.size": 14, "axes.labelsize": 16,
    "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 12,
    "font.weight": "bold", "axes.labelweight": "bold",
    "axes.linewidth": 1.6, "axes.edgecolor": "black",
    "xtick.major.width": 1.6, "ytick.major.width": 1.6,
    "xtick.major.size": 6, "ytick.major.size": 6,
    "xtick.color": "black", "ytick.color": "black",
    "xtick.direction": "in", "ytick.direction": "in",
    "lines.linewidth": 2.2, "lines.markersize": 7,
    "legend.frameon": True, "legend.edgecolor": "black",
    "figure.dpi": 100,        # match Colab's browser CSS pixel scale
})


def _percentile_stretch(a, lo=2, hi=98):
    """2-98 percentile stretch to [0, 1] for display; NaN-safe."""
    p_lo, p_hi = np.nanpercentile(a, [lo, hi])
    return np.clip((a - p_lo) / max(1e-9, p_hi - p_lo), 0, 1)


def _log_stretch(a):
    """log10 of positive SAR backscatter, NaN elsewhere, then percentile stretch."""
    out = np.where(a > 0, np.log10(a + 1e-6), np.nan)
    return _percentile_stretch(out)


## Sanity-check the AOI on OpenStreetMap

Before fetching multi-GB SAR + optical + DEM data, look at where the AOI actually is. The `+` marker is your target hint from the setup cell above; the `x` is the AOI centre that the NISAR granule-footprint picker resolved to (may differ from the hint if the picker re-centred on the best-covered granule's centroid). If neither is where you expected, tweak `TARGET_LAT` / `TARGET_LON` / `RADIUS_KM` in the setup cell and re-run.

In [ ]:
# AOI on OpenStreetMap -- sanity-check what we're actually looking at
# BEFORE we spend 5-10 minutes fetching multi-GB SAR data over the
# wrong bit of coastline. High-latitude AOIs are especially easy to
# miscentre (coordinates near the pole get numerically tricky), and
# NISAR's auto-picker can silently re-centre if your hint has poor
# coverage -- so a quick OSM view is worth it.
try:
    import contextily as cx
    HAVE_CTX = True
except ImportError:
    HAVE_CTX = False
    import subprocess as _sp, sys as _sys
    _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "contextily"])
    import contextily as cx
    HAVE_CTX = True

fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
# Show a wider frame than the AOI itself so context is visible.
pad = 0.5   # degrees of padding around the AOI for context
ax.set_xlim(AOI[0] - pad, AOI[2] + pad)
ax.set_ylim(AOI[1] - pad, AOI[3] + pad)
ax.set_xlabel(f"AOI centre ({aoi_lat:.3f}, {aoi_lon:.3f})  |  radius {RADIUS_KM:.1f} km  |  target: {TARGET_NAME}")
ax.set_ylabel("latitude [deg]")
ax.set_aspect(1.0 / np.cos(np.deg2rad(aoi_lat)))

# Draw the AOI rectangle so its footprint is unambiguous.
from matplotlib.patches import Rectangle
ax.add_patch(Rectangle((AOI[0], AOI[1]), AOI[2]-AOI[0], AOI[3]-AOI[1],
                        fill=False, edgecolor="#0072B2", lw=2.2,
                        label="fetched AOI"))
# Markers: target hint (user's request) + resolved AOI centre (may
# differ if the picker re-centred on a granule centroid).
ax.plot(TARGET_LON, TARGET_LAT, marker="+", markersize=18, mew=3,
        color="#D55E00", label=f"target hint")
ax.plot(aoi_lon, aoi_lat, marker="x", markersize=14, mew=3,
        color="#0072B2", label="AOI centre (fetched)")
ax.legend(loc="upper right", fontsize=10)

if HAVE_CTX:
    try:
        cx.add_basemap(ax, crs="EPSG:4326",
                       source=cx.providers.OpenStreetMap.Mapnik,
                       attribution_size=6)
    except Exception as e:
        print(f"(OSM basemap unavailable: {e}; falling back to plain grid)")
        ax.grid(True, ls=":", alpha=0.4)
else:
    ax.grid(True, ls=":", alpha=0.4)

plt.show()
plt.close(fig)


## Fetch NISAR L-band SAR

**NISAR** (NASA-ISRO SAR) is the first dual-frequency L+S-band SAR
mission with a public data archive. NASA released the L-band
(24 cm wavelength) L2 Geocoded Polarimetric Covariance (GCOV)
product publicly on **2026-07-20**; the S-band data from ISRO is
currently email-request only through the Bhoonidhi portal (see
`docs/providers/earthdata.md`).

Why L-band matters over ice: at 24 cm the microwave penetrates
substantially further into dry snow and firn than Sentinel-1's
C-band (5.6 cm), so the return you see on a glacier surface is a
mix of surface and shallow-subsurface scattering. This is the first
proper L-band archive since ALOS PALSAR-1 (2006-2011); the product
name carries the `PROVISIONAL` tag while the mission's calibration
window is still open.

The pipeline routes `"NISAR-L"` through the `earthdata` provider
via `PROVIDER_AUTO`; the provider queries NASA's Common Metadata
Repository (CMR) with `earthaccess`, downloads the L2 GCOV HDF5,
extracts the requested polarisations, and writes a
`NISAR-L_full_size.tiff` in the pipeline's canonical
per-scene layout.

**Polarisation availability recap:** NISAR observation modes vary
per pass. The current 7-week provisional archive has good dual-pol
(`DHDH` = HH+HV) coverage over northern Baffin Island (this
notebook's default), central and southern Canada, and Europe;
single-pol (`SHNA` = HH, `NASV` = VV) elsewhere. The picker prefers
dual-pol candidates when they exist; if none are available for your
AOI, you'll see `[HV] not present in this granule; filling with NaN`
in the fetcher output below. That's a NISAR archive characteristic,
not a pipeline bug — will improve as NISAR fills its 12-day repeat
cycle over 2026-2027.


In [ ]:
from geoai_datacubes.fetch import fetch_sentinel_data

t0 = time.time()
nisar_data, nisar_bands = fetch_sentinel_data(
    "NISAR-L",
    bands=["HH", "HV"],           # dual-pol is the typical acquisition
    time_range=TIME_RANGE,
    roi=AOI,
    resolution=RESOLUTION,
    save_folder=str(DATA),
    provider="auto",              # -> earthdata
)
nisar_scene = sorted(DATA.glob("NISAR-L_*"), key=os.path.getmtime)[-1]
print(f"\nNISAR-L fetched in {time.time()-t0:.1f}s")
print(f"scene folder : {nisar_scene.relative_to(REPO_ROOT)}")
print(f"bands        : {nisar_bands}")
print(f"array shape  : {nisar_data[0].shape}")


## Fetch Sentinel-1 (C-band SAR)

Sentinel-1 is ESA's C-band (5.6 cm) SAR workhorse -- the same physical
observable as NISAR-L but at ~4x shorter wavelength. Over a glacier
tongue the two disagree on purpose: C-band returns are dominated by
surface roughness (ice crevasses, meltwater ponding, wind-textured
fjord water), while L-band penetrates further into dry snow / firn and
mixes surface with shallow subsurface scattering. Pulling both into
the same UTM cube makes that dependence visible band-by-band.

`provider="auto"` routes S1 to `planetary_computer` (Microsoft's STAC
GRD archive), which requires no auth. **Polarisation choice matters
by region**: over most land Sentinel-1 IW acquires `VV+VH`, but over
Greenland and Antarctica the standard product is **`HH+HV`** — same
polarisations NISAR uses here, giving us a direct co-polarisation
comparison L vs C.


In [ ]:
s1_scene = None
HAVE_S1  = False
try:
    t0 = time.time()
    s1_data, s1_bands = fetch_sentinel_data(
        "Sentinel-1",
        bands=["HH", "HV"],           # Arctic / polar standard polarisation
        time_range=S1_TIME_RANGE,     # widened separately -- see setup cell
        roi=AOI,
        resolution=RESOLUTION,
        save_folder=str(DATA),
        provider="auto",              # -> planetary_computer
    )
    s1_scene = sorted(DATA.glob("Sentinel-1_*"), key=os.path.getmtime)[-1]
    HAVE_S1  = True
    print(f"\nSentinel-1 fetched in {time.time()-t0:.1f}s")
    print(f"scene folder : {s1_scene.relative_to(REPO_ROOT)}")
    print(f"bands        : {s1_bands}")
except Exception as exc:
    # Common causes: Colab-to-Azure transient timeout, no dual-pol RTC
    # in window, or "No scenes matched" on a marginal AOI. Downstream
    # cells check HAVE_S1 and skip S1 panels + L-vs-C comparison
    # gracefully if the fetch failed.
    print(f"Sentinel-1 fetch failed: {type(exc).__name__}: {exc}")
    print("Continuing without S1 -- rerun the cell to try again if "
          "this looks transient.")
    HAVE_S1 = False


## Fetch a DEM: ArcticDEM (default) or Copernicus DEM

Two choices, toggle via `DEM_MISSION` in the setup cell above:

- **`ArcticDEM`** (default here — the demo runs in the Arctic
  where the domain applies): 32 m native mosaic v4.1 from the
  Polar Geospatial Center (PGC, U. of Minnesota — PI Ian Howat,
  Ohio State University). Built from sub-metre commercial optical
  stereo (WorldView / GeoEye) via SETSM. Higher resolution than
  Copernicus (10 m and 2 m mosaic tiles also live on the same S3
  bucket if you want the higher-resolution product) *and*
  time-versioned (v1 → v4.1 spans 2015-present), which matters over
  ice tongues that change on decadal timescales. Arctic-only
  (> 60°N).
- **`Copernicus-DEM`** (global fallback): 30 m Tandem-X InSAR mosaic.
  Same product as notebook 04 uses; global, static single snapshot.
  Use this if your target AOI extends south of ~60°N or if you want
  the global-consistent baseline.

Both are static (`TIME_RANGE` ignored). Both write a
`<Mission>_full_size.tiff` in the pipeline's canonical per-scene
layout so the downstream fusion / visualisation code below doesn't
care which one was picked.


In [ ]:
# Apply ArcticDEM resolution knob from the setup cell. Only relevant
# when DEM_MISSION == "ArcticDEM"; no-op for Copernicus-DEM.
# NOTE: use set_arcticdem_resolution() rather than writing to the
# profile dict directly -- it updates release_tag at the same time,
# which is what names the output folder.
if DEM_MISSION == "ArcticDEM":
    from geoai_datacubes.fetch import set_arcticdem_resolution
    set_arcticdem_resolution(ARCTICDEM_RESOLUTION)

t0 = time.time()
dem_data, dem_bands = fetch_sentinel_data(
    DEM_MISSION,                       # "ArcticDEM" (default) or "Copernicus-DEM"
    bands=["DEM"],
    time_range=TIME_RANGE,             # ignored (static=True)
    roi=AOI,
    resolution=RESOLUTION,
    save_folder=str(DATA),
    provider="auto",                   # ArcticDEM -> direct_http; Copernicus -> earthsearch
)
dem_scene = sorted(DATA.glob(f"{DEM_MISSION}_*"), key=os.path.getmtime)[-1]
print(f"\n{DEM_MISSION} fetched in {time.time()-t0:.1f}s")
print(f"scene folder : {dem_scene.relative_to(REPO_ROOT)}")
print(f"bands        : {dem_bands}")


## Fetch Sentinel-2 (optical, cloud-permitting)

Sentinel-2 A/B fly a 5-day repeat that reaches most Arctic AOIs, but persistent
polar cloud + the mid-summer melt often leaves *no* cloud-free scene
in a 6-week window. The next cell wraps the fetch in `try / except` so
the notebook falls through gracefully to SAR + DEM only when there is
no usable optical scene. Switch `INCLUDE_S2 = False` in the setup cell
if you want to skip the attempt entirely.

`provider="auto"` routes S2 to `earthsearch` (AWS Element-84 STAC);
`max_cloud_coverage=MAX_CLOUD` picks the least-cloudy scene in the
window that meets the threshold.


In [ ]:
s2_scene = None
HAVE_S2  = False
if INCLUDE_S2:
    try:
        t0 = time.time()
        s2_data, s2_bands = fetch_sentinel_data(
            "Sentinel-2",
            bands=["B04", "B03", "B02"],
            time_range=S2_TIME_RANGE,   # widened separately -- see setup cell
            roi=AOI,
            resolution=RESOLUTION,
            save_folder=str(DATA),
            max_cloud_coverage=MAX_CLOUD,
            provider="auto",              # -> earthsearch
        )
        s2_scene = sorted(DATA.glob("Sentinel-2_*"), key=os.path.getmtime)[-1]
        HAVE_S2  = True
        print(f"\nSentinel-2 fetched in {time.time()-t0:.1f}s")
        print(f"scene folder : {s2_scene.relative_to(REPO_ROOT)}")
        print(f"bands        : {s2_bands}")
    except Exception as exc:
        print(f"No cloud-free S2 scene in window: {type(exc).__name__}: {exc}")
        print("Continuing without optical -- SAR + DEM only.")
        HAVE_S2 = False
else:
    print("INCLUDE_S2=False -- skipping Sentinel-2 fetch.")


## Visualise each modality separately

Before we fuse anything, look at each modality in isolation to
confirm the fetch worked and to build intuition about what the ML
cube will actually contain. SAR bands are shown greyscale, with a
per-band percentile stretch so a single bright specular return does
not wash out the whole panel. The DEM gets the standard NW-illuminated
hillshade so the glacier front and the surrounding relief pop out. S2
(if we got one) is a 2-98 percentile-stretched true-colour RGB.


In [ ]:
# NISAR-L HH + HV -- 2-panel percentile-stretched greyscale.
nisar_tiff = nisar_scene / "NISAR-L_full_size.tiff"
with rasterio.open(nisar_tiff) as src:
    descs = list(src.descriptions)
    nisar_hh = src.read(descs.index("HH") + 1).astype(np.float32) if "HH" in descs else None
    nisar_hv = src.read(descs.index("HV") + 1).astype(np.float32) if "HV" in descs else None

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
for ax, arr, name in zip(axes, [nisar_hh, nisar_hv], ["NISAR-L HH", "NISAR-L HV"]):
    if arr is None:
        ax.text(0.5, 0.5, f"{name}\nnot in scene", ha="center", va="center",
                transform=ax.transAxes, fontweight="bold")
        ax.set_xticks([]); ax.set_yticks([])
        continue
    ax.imshow(_percentile_stretch(arr), cmap="gray", interpolation="nearest")
    ax.set_xlabel(name)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect("equal")

plt.show()
plt.close(fig)   # release figure state so re-runs don't accumulate


In [ ]:
# Sentinel-1 HH + HV -- 2-panel log-stretched greyscale on the same
# canvas layout as the NISAR panels above so C-band vs L-band can be
# compared by eye at the SAME polarisation. GRD backscatter is
# strongly skewed, so we take log10 first before percentile-stretching.
if not HAVE_S1:
    print("HAVE_S1 = False, skipping S1 visualisation.")
else:
    s1_tiff = s1_scene / "Sentinel-1_full_size.tiff"
    with rasterio.open(s1_tiff) as src:
        descs = list(src.descriptions)
        s1_hh = src.read(descs.index("HH") + 1).astype(np.float32) if "HH" in descs else None
        s1_hv = src.read(descs.index("HV") + 1).astype(np.float32) if "HV" in descs else None

    def _log_stretch(a):
        """log10 of positive backscatter, NaN elsewhere, then percentile stretch."""
        out = np.where(a > 0, np.log10(a + 1e-6), np.nan)
        return _percentile_stretch(out)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)
    for ax, arr, name in zip(axes, [s1_hh, s1_hv], ["Sentinel-1 HH (log)", "Sentinel-1 HV (log)"]):
        if arr is None:
            ax.text(0.5, 0.5, f"{name}\nnot in scene", ha="center", va="center",
                    transform=ax.transAxes, fontweight="bold")
            ax.set_xticks([]); ax.set_yticks([])
            continue
        ax.imshow(_log_stretch(arr), cmap="gray", interpolation="nearest")
        ax.set_xlabel(name)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_aspect("equal")

    plt.show()
    plt.close(fig)   # release figure state so re-runs don't accumulate


In [ ]:
# DEM -- NW-illuminated hillshade with terrain colormap.
# Illuminate from the northwest, low in the sky (azdeg=315, altdeg=45)
# so north-facing slopes brighten and shadows fall southeast.
from matplotlib.colors import LightSource

dem_tiff = dem_scene / f"{DEM_MISSION}_full_size.tiff"
with rasterio.open(dem_tiff) as src:
    z = src.read(1).astype(np.float32)

ls = LightSource(azdeg=315, altdeg=45)
shaded = ls.shade(z, cmap=plt.cm.terrain, vert_exag=10, blend_mode="soft")

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
ax.imshow(shaded)
ax.set_xlabel(f"{DEM_MISSION}: elevation {np.nanmin(z):.0f} - {np.nanmax(z):.0f} m")
ax.set_xticks([]); ax.set_yticks([])
plt.show()
plt.close(fig)   # release figure state so re-runs don't accumulate


In [ ]:
# Sentinel-2 RGB (only if we got a cloud-free scene).
if HAVE_S2:
    s2_tiff = s2_scene / "Sentinel-2_full_size.tiff"
    with rasterio.open(s2_tiff) as src:
        descs = list(src.descriptions)
        r = src.read(descs.index("B04") + 1).astype(np.float32)
        g = src.read(descs.index("B03") + 1).astype(np.float32)
        b = src.read(descs.index("B02") + 1).astype(np.float32)

    rgb = np.dstack([_percentile_stretch(r),
                     _percentile_stretch(g),
                     _percentile_stretch(b)])

    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    ax.imshow(rgb)
    ax.set_xlabel("Sentinel-2 true colour (B04/B03/B02)")
    ax.set_xticks([]); ax.set_yticks([])
    plt.show()
    plt.close(fig)   # release figure state so re-runs don't accumulate
else:
    print("No cloud-free Sentinel-2 scene in window -- skipping RGB preview.")


## Fuse into a single data cube

`fuse_response_tiffs` reprojects every input onto a common UTM grid at
`resolution=RESOLUTION` and clips to the intersection footprint.
Resampling is band-kind-aware (see
`preprocessing.fusion._NEAREST_BANDS`): the SAR polarisation bands
(`HH`, `HV`, `VV`, `VH`) and continuous elevation use bilinear;
anything categorical would use nearest -- we have none in this cube.
Fused band names are prefixed with the source mission -- e.g.
`NISAR-L_HH`, `Sentinel-1_VV`, `Copernicus-DEM_DEM`,
`Sentinel-2_B04` -- which is what any downstream feature-selection
code keys on.


In [ ]:
from geoai_datacubes.preprocessing.fusion import fuse_response_tiffs

# Order matters only for CRS: first input's CRS becomes the target
# grid CRS unless `dst_crs=` is passed. NISAR-L is written by the
# earthdata provider in the polar-appropriate UTM zone for the AOI,
# so we put it first.
inputs = [
    str(nisar_scene / "NISAR-L_full_size.tiff"),
    str(dem_scene   / f"{DEM_MISSION}_full_size.tiff"),
]
if HAVE_S1:
    inputs.insert(1, str(s1_scene / "Sentinel-1_full_size.tiff"))
if HAVE_S2:
    inputs.append(str(s2_scene / "Sentinel-2_full_size.tiff"))

cube_path = OUT / "cube.tif"

t0 = time.time()
cube_meta = fuse_response_tiffs(
    inputs=inputs,
    output_path=str(cube_path),
    resolution=RESOLUTION,
    bbox_mode="intersection",
)
print(f"\nfused in {time.time()-t0:.1f}s")

with rasterio.open(cube_path) as src:
    cube_bands = list(src.descriptions)
    cube_shape = (src.count, src.height, src.width)
    cube_crs   = src.crs
print(f"bands  : {cube_bands}")
print(f"shape  : {cube_shape}")
print(f"CRS    : {cube_crs}")


## SAR wavelength comparison: L-band vs C-band on ice

Because the cube is now on a single UTM grid, we can put NISAR HH
and Sentinel-1 VV side by side pixel-for-pixel. Expect them to
*disagree* in a physically meaningful way:

- **L-band (NISAR, 24 cm)** penetrates further into dry snow and
  firn, so the glacier surface tends to look smoother -- the return
  is a mix of surface and shallow subsurface scattering, and small
  surface roughness gets averaged out.
- **C-band (Sentinel-1, 5.6 cm)** scatters more from the surface
  itself, so crevasses, meltwater ponding, and wind-textured fjord
  water all show up as brighter, more granular texture.

Both panels use the same per-band normalisation (a 2-98 percentile
stretch of the raw backscatter after log10), so absolute brightness
differences between them reflect the wavelength-dependent physics,
not a display artefact.


In [ ]:
# 2-panel side-by-side of NISAR HH and S1 HH, on the shared UTM grid --
# direct L-band vs C-band comparison AT THE SAME POLARISATION.
with rasterio.open(cube_path) as src:
    descs = list(src.descriptions)
    arr = src.read()


def _get(band_key):
    """Return the first band in the fused cube whose name ends in the
    given suffix (e.g. 'NISAR-L_HH'), or None if absent."""
    for i, d in enumerate(descs):
        if d and d.endswith(band_key):
            return arr[i].astype(np.float32)
    return None


nisar_hh_c = _get("NISAR-L_HH")
s1_hh_c    = _get("Sentinel-1_HH")

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5), constrained_layout=True)

if nisar_hh_c is None:
    axes[0].text(0.5, 0.5, "NISAR-L HH\nnot in cube",
                 ha="center", va="center",
                 transform=axes[0].transAxes, fontweight="bold")
else:
    axes[0].imshow(_log_stretch(nisar_hh_c), cmap="gray", interpolation="nearest")
axes[0].set_xlabel("NISAR-L HH (log stretch, L-band 24 cm)")
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].set_aspect("equal")

if s1_hh_c is None:
    axes[1].text(0.5, 0.5, "Sentinel-1 HH\nnot in cube",
                 ha="center", va="center",
                 transform=axes[1].transAxes, fontweight="bold")
else:
    axes[1].imshow(_log_stretch(s1_hh_c), cmap="gray", interpolation="nearest")
axes[1].set_xlabel("Sentinel-1 HH (log stretch, C-band 5.6 cm)")
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].set_aspect("equal")

plt.show()
plt.close(fig)   # release figure state so re-runs don't accumulate


## Where to go next

- **Track the calving front through the time series.** NISAR revisits
  the AOI every ~2-5 days at high latitudes; `TIME_RANGE` on the fetch call can
  be widened to a full melt season and looped over per-scene to
  animate the front position.
- **Add ALOS PALSAR-1 for pre-2011 L-band history.** The
  `planetary_computer` provider already carries ALOS PALSAR mosaics;
  chaining them onto the NISAR archive gives a ~20-year L-band record
  interrupted only by the 2011-2026 archive gap.
- **Higher-resolution ArcticDEM.** This notebook uses the 32 m
  ArcticDEM v4.1 mosaic; PGC also publishes 10 m and 2 m mosaics on
  the same S3 bucket. To swap, edit `_ARCTICDEM_RES` in
  `geoai_datacubes/fetch/missions.py` (currently a module constant;
  future work: expose as a mission-config field so it can be set
  from the notebook toggle).
- **Fuse in an ice-velocity product.** NSIDC's MEaSUREs Greenland Ice
  Velocity Map lives in the same NASA Earthdata cloud that this
  notebook already authenticates against -- the `earthdata` provider
  path just needs the product's `short_name` added to the mission
  registry.
- **Move the demo to another target.** Swap `TARGET_LAT` /
  `TARGET_LON` in the setup cell to Sermeq Kujalleq / Jakobshavn
  (69.15 N, -50.0 W) or Zachariae Isstrom (79.0 N, -20.0 W) — both
  fast-flowing tidewater glaciers with active calving fronts. The
  `_pick_nisar_covered_aoi` helper will find you a good NISAR
  granule automatically.
- **Try the S-band data if ISRO opens automated access.** NISAR's
  S-band L2 products are currently email-request only through
  ISRO's Bhoonidhi portal (see `docs/providers/earthdata.md`); once
  they land in CMR the same `fetch_sentinel_data("NISAR-S", ...)`
  call will Just Work.
